* [#57](https://github.com/salgo60/spa2Commons/issues/57)
* this Notebook [57_SPA_WD_graph.ipynb](57_SPA_WD_graph.ipynb)
  * rapport [spa_graph.html](https://salgo60.github.io/spa2Commons/Notebook/spa_graph.html) 

In [1]:
from datetime import datetime
start_time  = datetime.now()
print("Last run: ", start_time)

Last run:  2026-03-09 12:31:30.733738


In [2]:
#!pip install requests pandas networkx pyvis SPARQLWrapper rdflib tqdm

In [3]:

# ===============================
# SPA KNOWLEDGE GRAPH POC
# Wikidata + Commons + SPA API
# ===============================

import requests
import pandas as pd
import networkx as nx
from pyvis.network import Network
from SPARQLWrapper import SPARQLWrapper, JSON
from tqdm import tqdm
import time

# -------------------------------
# SESSION + USER AGENT
# -------------------------------

USER_AGENT = "SPA-Graph-POC/1.0 (https://github.com/salgo60; mailto:salgo60@msn.com)"

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept": "application/json"
})

# -------------------------------
# SPARQL HELPER
# -------------------------------

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

def run_sparql(query):

    sparql.setQuery(query)
    return sparql.query().convert()


# -------------------------------
# HÄMTA PERSONER MED SPA-ID
# -------------------------------

print("Fetching SPA IDs from Wikidata...")

query = """
SELECT ?person ?personLabel ?spa_id WHERE {
  ?person wdt:P4819 ?spa_id .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "sv,en". }
}
LIMIT 500
"""

data = run_sparql(query)

records = []

for row in data["results"]["bindings"]:

    qid = row["person"]["value"].split("/")[-1]

    records.append({
        "qid": qid,
        "name": row["personLabel"]["value"],
        "spa_id": row["spa_id"]["value"]
    })

df = pd.DataFrame(records)

print("Persons with SPA id:", len(df))


# -------------------------------
# HÄMTA COMMONS-BILDER
# -------------------------------

print("Fetching Commons images...")

commons_query = """
SELECT ?file ?spa_id WHERE {
  ?file wdt:P4819 ?spa_id .
}
LIMIT 500
"""

commons_data = run_sparql(commons_query)

commons_map = {}

for row in commons_data["results"]["bindings"]:

    spa_id = row["spa_id"]["value"]
    file_url = row["file"]["value"]

    commons_map.setdefault(spa_id, []).append(file_url)


print("Commons images:", len(commons_map))


# -------------------------------
# SPA API
# -------------------------------

def spa_portrait(spa_id):

    url = "https://portrattarkiv.se/endpoints/portraits.php"

    try:

        r = session.get(url, params={"id": spa_id})

        if r.status_code == 200:
            return r.json()

    except:
        pass

    return None


# -------------------------------
# BUILD GRAPH
# -------------------------------

print("Building graph...")

G = nx.Graph()

for _, row in tqdm(df.iterrows(), total=len(df)):

    person = row["name"]
    spa_id = row["spa_id"]

    portrait_node = f"SPA:{spa_id}"

    G.add_node(person, type="person")
    G.add_node(portrait_node, type="portrait")

    G.add_edge(person, portrait_node, relation="depicted_in")

    # Commons images
    if spa_id in commons_map:

        for file in commons_map[spa_id]:

            image_node = f"Commons:{file.split('/')[-1]}"

            G.add_node(image_node, type="image")

            G.add_edge(portrait_node, image_node, relation="has_image")

    # SPA metadata
    data = spa_portrait(spa_id)

    if data:

        place = data.get("place")

        if place:

            G.add_node(place, type="place")
            G.add_edge(portrait_node, place, relation="place")

    time.sleep(0.1)


print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))


# -------------------------------
# VISUALISERA GRAF
# -------------------------------

print("Creating visualization...")

net = Network(height="800px", width="100%", bgcolor="#111111", font_color="white")

for node, data in G.nodes(data=True):

    if data["type"] == "person":
        net.add_node(node, color="lightblue")

    elif data["type"] == "portrait":
        net.add_node(node, color="orange")

    elif data["type"] == "image":
        net.add_node(node, color="green")

    else:
        net.add_node(node, color="gray")


for source, target, data in G.edges(data=True):

    net.add_edge(source, target, title=data["relation"])


net.write_html("spa_graph.html") 

print("Graph saved as spa_graph.html")


Fetching SPA IDs from Wikidata...
Persons with SPA id: 500
Fetching Commons images...
Commons images: 500
Building graph...


100%|█████████████████████████████████████████| 500/500 [01:23<00:00,  5.97it/s]


Nodes: 1395
Edges: 900
Creating visualization...
spa_graph.html


AttributeError: 'NoneType' object has no attribute 'render'

In [4]:
net.write_html("spa_graph.html")

In [ ]:
from datetime import datetime
end = datetime.now()
print("Ended: ", end) 
print('Time elapsed (hh:mm:ss.ms) {}'.format(datetime.now() - start_time))